In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
bronze_path = f"{s3_root_path}/Bronze/Nisarga/results"
silver_path = f"{s3_root_path}/Silver/Nisarga/results"
 
print("Bronze Path :", bronze_path)
print("Silver Path :", silver_path)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 1 - READ BRONZE RESULTS DATA
# ============================================================
 
results_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)
 
display(results_df)
 
 
# COMMAND ----------
 
from pyspark.sql import functions as F
 
 
# COMMAND ----------
 
# ============================================================
# STEP 2 - SELECT REQUIRED COLUMNS
# ============================================================
 
results_selected_df = results_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("driverId"),
    F.col("constructorId"),
    F.col("number"),
    F.col("grid"),
    F.col("position"),
    F.col("positionText"),
    F.col("points"),
    F.col("laps"),
    F.col("status"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)
 
display(results_selected_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 3 - STANDARDIZE COLUMN NAMES
# ============================================================
 
results_renamed_df = (
    results_selected_df
    .withColumnsRenamed({
        "raceName": "race_name",
        "driverId": "driver_id",
        "constructorId": "constructor_id",
        "positionText": "position_text"
    })
)
 
display(results_renamed_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 4 - REMOVE NULL BUSINESS KEYS
# ============================================================
 
results_valid_df = (
    results_renamed_df
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("driver_id").isNotNull() &
        F.col("constructor_id").isNotNull()
    )
)
 
display(results_valid_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 5 - REMOVE DUPLICATES
# ============================================================
# One driver should have one result for a particular race.
 
results_distinct_df = (
    results_valid_df
    .dropDuplicates([
        "season",
        "round",
        "driver_id"
    ])
)
 
display(results_distinct_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 6 - STANDARDIZE TEXT VALUES
# ============================================================
 
results_clean_df = (
    results_distinct_df
    .withColumn(
        "race_name",
        F.initcap(F.trim(F.col("race_name")))
    )
)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 7 - CREATE DERIVED COLUMN
# ============================================================
# is_winner = True when finishing position is 1.
 
results_final_df = (
    results_clean_df
    .withColumn(
        "is_winner",
        F.when(F.col("position") == 1, True)
         .otherwise(False)
    )
)
 
display(results_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 8 - WRITE TO SILVER
# ============================================================
 
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 9 - VALIDATE SILVER DATA
# ============================================================
 
silver_results_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)
 
display(silver_results_df)
 
print("Bronze Records :", results_df.count())
print("Silver Records :", silver_results_df.count())
 
print("Silver Results data successfully written to:")
print(silver_path)